# HRM-Text-1B in KerasHub

This notebook converts the pinned Apache-2.0 `sapientinc/HRM-Text-1B` checkpoint to KerasHub in memory, verifies Keras logits against Transformers, and runs greedy cached generation. Use a GPU runtime (a T4 is sufficient for the short sequence used here).

The official checkpoint is a pre-alignment PrefixLM model, not a chat model. The generation cell marks the prompt as a bidirectional PrefixLM prefix, which matches pretraining-time prefill semantics.

In [ ]:
import json
import os
import subprocess

# This immutable commit contains the implementation under review. After
# merge, replace these with the upstream repository and its merged commit.
REPOSITORY = "https://github.com/pzarzycki/keras-hub.git"
KERAS_HUB_REVISION = "5ff84000167698f0c412fb9cb8121c878e932749"
WORKDIR = "/content/keras-hub"

subprocess.run(["bash", "-lc", "command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
subprocess.run(["git", "clone", REPOSITORY, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "checkout", "--detach", KERAS_HUB_REVISION], check=True)
# Pin Transformers without globally upgrading Colab's compiled scientific stack.
subprocess.run(["uv", "pip", "install", "--system", "-q", "transformers==5.9.0", "safetensors"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", WORKDIR], check=True)

# Keras must select its backend before it is imported.
os.environ["KERAS_BACKEND"] = "torch"


In [ ]:
import os
os.chdir(WORKDIR)

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before continuing.")

torch.set_default_device("cuda")
import keras
from keras_hub.src.models.hrm_text.hrm_text_causal_lm import HrmTextCausalLM
from tools.checkpoint_conversion.convert_hrm_text_checkpoints import (
    convert_tokenizer,
    convert_weights,
    create_backbone,
)

MODEL_ID = "sapientinc/HRM-Text-1B"
REVISION = "9f082d68b8cd0ebc56e33f1c88c45609174c272c"
keras.config.set_dtype_policy("float32")
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float32
).cuda().eval()

backbone = create_backbone(hf_model.config)
convert_weights(backbone, hf_model)
tokenizer = convert_tokenizer(hf_tokenizer)
model = HrmTextCausalLM(backbone, preprocessor=None)
print(f"Keras backend: {keras.config.backend()}")
print(f"Logical KV-cache slots: {backbone.cache_slots}")


## Numerical verification

The conversion rejects any unmapped tensor. This check then compares all logits for a short input. The expected maximum absolute error is below `2e-4`; the initial port measured `1.29e-5` on a T4.

In [ ]:
text = "HRM-Text validates a portable Keras implementation."
hf_inputs = hf_tokenizer(text, return_tensors="pt").to("cuda")
hf_inputs["token_type_ids"] = torch.zeros_like(hf_inputs["input_ids"])
with torch.inference_mode():
    hf_logits = hf_model(**hf_inputs).logits.float().cpu().numpy()

keras_inputs = {
    "token_ids": hf_inputs["input_ids"].cpu().numpy(),
    "padding_mask": hf_inputs["attention_mask"].cpu().numpy(),
    "token_type_ids": hf_inputs["token_type_ids"].cpu().numpy(),
}
keras_logits = keras.ops.convert_to_numpy(model(keras_inputs))
max_abs_error = np.max(np.abs(keras_logits - hf_logits))
np.testing.assert_allclose(keras_logits, hf_logits, atol=2e-4, rtol=2e-4)
print(f"Forward parity passed; max absolute logit error: {max_abs_error:.3e}")


## PrefixLM generation

The full prompt is a PrefixLM condition block (`token_type_ids == 1`). `generate()` uses the model's logical recurrent KV cache internally; the 1B configuration has `16 × 2 × (3 + 1) = 128` cache slots. Output quality is intentionally not evaluated here because this is a base, pre-alignment checkpoint.

In [ ]:
condition = "<|quad_end|><|object_ref_end|>"  # synth,cot
prompt = f"<|im_start|>{condition}What is the capital of France?<|im_end|>"
prompt_ids = hf_tokenizer(prompt, add_special_tokens=False)["input_ids"]
# Four tokens are enough to exercise cached decoding on a T4 runtime.
max_length = len(prompt_ids) + 4
token_ids = np.full((1, max_length), tokenizer.pad_token_id, dtype="int32")
token_ids[0, : len(prompt_ids)] = prompt_ids
padding_mask = np.zeros((1, max_length), dtype="int32")
padding_mask[0, : len(prompt_ids)] = 1
prefix_mask = padding_mask.copy()

model.compile(sampler="greedy")
generated = model.generate(
    {
        "token_ids": token_ids,
        "padding_mask": padding_mask,
        "token_type_ids": prefix_mask,
    },
    max_length=max_length,
    stop_token_ids=None,
)
generated_text = tokenizer.detokenize(generated["token_ids"][0])
print(generated_text)
with open("/content/hrm_text_1b_results.json", "w") as f:
    json.dump(
        {
            "max_abs_logit_error": float(max_abs_error),
            "cache_slots": backbone.cache_slots,
            "generated_text": str(generated_text),
        },
        f,
    )


## Optional: save a reusable local Keras preset

Uncomment the next line after validation to save the converted model and tokenizer. The resulting directory can be loaded with `keras_hub.models.HrmTextCausalLM.from_preset(path)`.

In [ ]:
# model.save_to_preset("/content/hrm_text_1b")
